## Filtering Presentation Types

### What can you do with this Notebook?

This notebook explores some techniques for filtering the reults of the Presentation Type tool.  See the main notebook for Presentation types to review the various options.



In [1]:
import crim_intervals
from crim_intervals import * 
from crim_intervals import main_objs
import pandas as pd
import re
import os
import numpy
import itertools
from music21 import *
MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)
else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)
else:
    print(MUSDIR, "folder already exists.")

saved_csv folder already exists.
Music_Files folder already exists.


In [2]:
# from music21 import converter

# volpiano_stream = music21.converter.parse('./sample.volpiano')

# part = music21.converter.parse

In [6]:
# from music21.converter.subConverters import ConverterVolpiano

volpiano_string = '1---d--dc---f--g---f--gh--h---hkjklk--hijh7---h---gf--ghjh-hg--hg---g---g--e--g---hg7--fg---fe--c---eg--e---fe--d--d---4---h--h--g--f--gh--g7---3'
test = converter.parse('volpiano: ' + volpiano_string)

# Add metadata
test.metadata = metadata.Metadata()
test.metadata.title = 'ID123'
test.metadata.composer = 'Regensburg'

# Add part name
p = stream.Part()
p.partName = 'Cantus'
test.insert(0, p)

# Display the metadata
print(f"Title: {test.metadata.title}")
print(f"Composer: {test.metadata.composer}")
print(f"Part Name: {p.partName}")


test.write('musicxml', 'test.xml')

Title: ID123
Composer: Regensburg
Part Name: Cantus


PosixPath('/Users/rfreedma/Documents/CRIM_Python/CRIM_JHUB/test.xml')

In [11]:
piece = importScore('test.xml')
nr = piece.notes()
nr.index.rename('Index', inplace=True)
nr.rename(columns={'Part-1': 'cantus'}, inplace=True)
print(piece.metadata)
mel = piece.melodic(df = nr, kind='d')
ng = piece.ngrams(df=mel, n=4)
ng.value_counts().to_frame()


{'title': 'ID123', 'composer': 'Regensburg', 'date': None}


,0
cantus,
"(-2, 2, 2, -2)",2
"(-2, -2, 2, 2)",2
"(1, -2, -2, 2)",2
"(2, -2, 1, -2)",2
"(2, 1, 1, 3)",1
"(1, 5, 1, -2)",1
"(2, -2, -2, -3)",1
"(2, -2, -2, 1)",1
"(2, -2, -2, 2)",1


#### Load the Piece Here

* Note that you can load from CRIM, or put a file in the **Music_Files** folder in the Notebook.

In [76]:
# Select a prefix:
# prefix = 'Music_Files/'
# prefix = 'https://raw.githubusercontent.com/RichardFreedman/CRIM_Additions/main/'
prefix = 'https://crimproject.org/mei/'

# add the CRIM Piece ID here
mei_file = 'CRIM_Model_0008.mei'
url = prefix + mei_file
piece = importScore(url)

# print the details about composer and title to be sure you have loaded the correct piece
print(piece.metadata)


{'title': 'Ave Maria', 'composer': 'Josquin Des Prés', 'date': 1502}


In [78]:
piece.melodic()


,cantus,Part-2,Part-3,Part-4
0.0,NaN,Rest,Rest,Rest
4.0,P4,NaN,NaN,NaN
8.0,NaN,Rest,Rest,Rest
12.0,P1,NaN,NaN,NaN
16.0,M2,NaN,Rest,Rest
...,...,...,...,...
1248.0,-M2,-m3,m2,M2
1252.0,NaN,m2,NaN,NaN
1256.0,-m2,M2,M2,-M2
1272.0,m2,P1,-M2,-P5


## Find Presentation Types
* `piece.presentationTypes()`

- **limit to entries** (following rests or section) = `limit_to_entries = True`.
- allowing **'moving window'** of all patterns in every voice = `limit_to_entries = False`
- set the **length of the soggetti** with `melodic_ngram_length = n`
- set **flexed threshold for first interval** `flex_threshold=1`
- set the **maximum difference between similar soggetti** with `edit_distance_threshold = n`
- to **include all the hidden PENs and IDS** (those found within longer Fugas, use `include_hidden_types = True`.  
- for faster (and simpler) listing of points of imitation **without hidden forms**, use `include_hidden_types = False`

* For example:

```
piece.presentationTypes(limit_to_entries = True,
                            body_flex = 0,
                            head_flex = 1,
                            include_hidden_types = False,
                            combine_unisons = True,
                            melodic_ngram_length = 4)
```
                       
                       
* Read the documentation:  `print(piece.presentationTypes.__doc__)`

In [4]:
p_types = piece.presentationTypes(limit_to_entries = False,
                        body_flex = 0,
                        head_flex = 1,
                        include_hidden_types = False,
                        combine_unisons = True,
                        melodic_ngram_length = 4)

In [5]:
# show the full results
pd.set_option('display.max_rows', None)
p_types


,Composer,Title,First_Offset,Measures_Beats,Melodic_Entry_Intervals,Offsets,Soggetti,Time_Entry_Intervals,Voices,Presentation_Type,Number_Entries,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps,Progress
0,Josquin Des Prés,Ave Maria,0.0,"[1/1.0, 3/1.0, 5/1.0, 7/1.0]","[P-8, P1, P-8]","[0.0, 16.0, 32.0, 48.0]","[(4, 2, 2, -3), (3, 2, 2, -3)]","[16.0, 16.0, 16.0]","[[Superius], Altus, Tenor, Bassus]",PEN,4,True,False,None,0,0.000000
1,Josquin Des Prés,Ave Maria,0.0,"[1/1.0, 3/1.0, 5/1.0, 7/1.0]","[P-8, P1, P-8]","[0.0, 16.0, 32.0, 48.0]","[(4, 2, 2, -3), (3, 2, 2, -3), (2, 2, 2, -3)]","[16.0, 16.0, 16.0]","[[Superius], Altus, Tenor, Bassus]",PEN,4,True,False,None,0,0.000000
2,Josquin Des Prés,Ave Maria,56.0,"[8/1.0, 10/1.0, 12/1.0, 14/1.0]","[P-8, P1, P-8]","[56.0, 72.0, 88.0, 104.0]","[(-2, -2, -2, 2), (-3, -2, -2, 2)]","[16.0, 16.0, 16.0]","[[Superius], Altus, Tenor, Bassus]",PEN,4,True,False,None,3,0.043478
3,Josquin Des Prés,Ave Maria,62.0,"[8/4.0, 10/4.0, 12/4.0]","[P-8, P1]","[62.0, 78.0, 94.0]","[(-2, -2, 2, -2)]","[16.0, 16.0]","[[Superius], Altus, Tenor]",PEN,3,False,False,None,2,0.048137
4,Josquin Des Prés,Ave Maria,64.0,"[9/1.0, 11/1.0, 13/1.0]","[P-8, P1]","[64.0, 80.0, 96.0]","[(-2, 2, -2, 4)]","[16.0, 16.0]","[[Superius], Altus, Tenor]",PEN,3,False,False,None,2,0.049689
5,Josquin Des Prés,Ave Maria,68.0,"[9/3.0, 11/3.0, 13/3.0]","[P-8, P1]","[68.0, 84.0, 100.0]","[(2, -2, 4, -2), (3, -2, 4, -2)]","[16.0, 16.0]","[[Superius], Altus, Tenor]",PEN,3,True,False,None,2,0.052795
6,Josquin Des Prés,Ave Maria,72.0,"[10/1.0, 12/1.0, 14/1.0]","[P-8, P1]","[72.0, 88.0, 104.0]","[(-2, 4, -2, -2), (-3, 4, -2, -2)]","[16.0, 16.0]","[[Superius], Altus, Tenor]",PEN,3,True,False,None,2,0.055901
7,Josquin Des Prés,Ave Maria,74.0,"[10/2.0, 12/2.5, 14/2.0]","[P-8, P1]","[74.0, 91.0, 106.0]","[(4, -2, -2, 2), (3, -2, -2, 2), (5, -2, -2, 2)]","[17.0, 15.0]","[[Superius], Altus, Tenor]",FUGA,3,True,False,None,2,0.057453
8,Josquin Des Prés,Ave Maria,74.0,"[10/2.0, 12/2.5, 14/2.0]","[P-8, P1]","[74.0, 91.0, 106.0]","[(4, -2, -2, 2), (2, -2, -2, 2), (3, -2, -2, 2)]","[17.0, 15.0]","[[Superius], Altus, Tenor]",FUGA,3,True,False,None,2,0.057453
9,Josquin Des Prés,Ave Maria,74.0,"[10/2.0, 12/2.5, 14/2.0]","[P-8, P1]","[74.0, 91.0, 106.0]","[(4, -2, -2, 2), (5, -2, -2, 2)]","[17.0, 15.0]","[[Superius], Altus, Tenor]",FUGA,3,True,False,None,2,0.057453


In [7]:
tuple_to_match = ('4', '2', '2', '-3')
filtered_df = p_types[p_types['Soggetti'].apply(lambda x: tuple_to_match in x)]
filtered_df

,Composer,Title,First_Offset,Measures_Beats,Melodic_Entry_Intervals,Offsets,Soggetti,Time_Entry_Intervals,Voices,Presentation_Type,Number_Entries,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps,Progress
0,Josquin Des Prés,Ave Maria,0.0,"[1/1.0, 3/1.0, 5/1.0, 7/1.0]","[P-8, P1, P-8]","[0.0, 16.0, 32.0, 48.0]","[(4, 2, 2, -3), (3, 2, 2, -3)]","[16.0, 16.0, 16.0]","[[Superius], Altus, Tenor, Bassus]",PEN,4,True,False,None,0,0.000000
1,Josquin Des Prés,Ave Maria,0.0,"[1/1.0, 3/1.0, 5/1.0, 7/1.0]","[P-8, P1, P-8]","[0.0, 16.0, 32.0, 48.0]","[(4, 2, 2, -3), (3, 2, 2, -3), (2, 2, 2, -3)]","[16.0, 16.0, 16.0]","[[Superius], Altus, Tenor, Bassus]",PEN,4,True,False,None,0,0.000000
23,Josquin Des Prés,Ave Maria,316.0,"[40/3.0, 44/3.0, 44/4.0, 46/2.0, 46/3.0, 46/4....","[P5, m-10, m6, M6, M-10, M2]","[316.0, 348.0, 350.0, 362.0, 364.0, 366.0, 402.0]","[(4, 2, 2, -3), (3, 2, 2, -3), (2, 2, 2, -3)]","[32.0, 2.0, 12.0, 2.0, 2.0, 36.0]","[Altus, [Superius], Bassus, Tenor, [Superius],...",FUGA,7,True,True,Tenor,3,0.245342


## Filtering the Results

Pandas offers many ways to filter any dataframe.  One of the simplest involves checking a single column for exact or partial matches..

**Exact Match the Full Contents of a Cell.** For this we use the Python method `isin(['My_Exact_Pattern'])`, which returns `True` for any row in which the given columns is an **exact match** of the given string.  If you run the following in a cell, you will see a long Series of True or False values.

```
p_types["Presentation_Type"].isin(['PEN'])
```
    
We can in turn use this **Boolean Series** to "mask" the entire dataframe to show either all the rows for which the condition is True or False (and so we could show all the PEN's, or everything *except* the PEN's.
    
```p_types[p_types["Presentation_Type"].isin(['PEN'])]
```
    
NOT the PENs includes the "~" to reverse the True and False values:
    
```
p_types[~p_types["Presentation_Type"].isin(['PEN'])]
```

You can in fact pass several possible values in this way.  But whether you are searching for one or more strings, they must be presented in the form of a Python **list**:  `['My_Exact_Pattern_1', 'My_Exact_Pattern_2']`.  For instance, we could  look for either **PEN or ID**:
    
```
p_types[p_types["Presentation_Type"].isin(['PEN', 'ID'])]
```
    
Finally, note that a **partial match** would return no results:
    
```
p_types[p_types["Presentation_Type"].isin(['P'])]
```


**Partial Match of Characters** is possible with `str.contains` method.  This returns `True` for any row in which our search pattern is *contained anywhere in the given cell*.  We could find any row in which the column "Presentation Type" contains "P" *anywhere, and not just a complete exact match*:

```
p_types[p_types["Presentation_Type"].str.contains("P")]
```
    
In practice it's useful to give such searches a name of their own, and to make a `.copy()` of the dataframe so that you don't disturb the original results:
    
```
pens_only = p_types[p_types["Presentation_Type"].str.contains("P")].copy()
```

Of course it would be more interesting to search for substrings in something like the Melodic Intervals column (which have various sets of values where we might want to know which patterns contain a M3 in the midst of some other series of entries).  But for this we need to manage the data types (see below).
    
**Comparisons:  < or >** can be useful for Fuga, as when you might want to find all the Fugas longer (or shorter) than a certain size. This code, for instance, will return a Boolean Series (True/False) of the column for Number_Entries:

```
p_types['Number_Entries'] < 4
```
    
   
We can in turn use this to 'mask' the p_types dataframe itself (notice the two sets of brackets):

```
p_types[p_types['Number_Entries'] < 4]
```

Or the opposite of the Boolean Series, so that we now have all the Fugas *not* less than 4:

```p_types[~p_types['Number_Entries'] < 4]
```


In [1]:
# Try things yourself!
p_types[p_types["Presentation_Type"].str.contains("ID")]

NameError: name 'p_types' is not defined

In [7]:
p_types[p_types['Number_Entries'] > 4]
       

,Composer,Title,First_Offset,Measures_Beats,Melodic_Entry_Intervals,Offsets,Soggetti,Time_Entry_Intervals,Voices,Presentation_Type,Number_Entries,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps
7,Josquin Des Prés,Ave Maria,76.0,"[10/3.0, 12/3.0, 14/3.0, 14/4.0, 23/1.0, 28/1....","[P-8, P1, m-9, m9, P-8, M9]","[76.0, 92.0, 108.0, 110.0, 176.0, 216.0, 221.0]","[(-2, -2, 2, 2), (-3, -2, 2, 2)]","[16.0, 16.0, 2.0, 66.0, 40.0, 5.0]","[[Superius], Altus, Tenor, Bassus, Altus, Bass...",FUGA,7,True,0.0,None,2
8,Josquin Des Prés,Ave Maria,140.0,"[18/3.0, 19/1.0, 19/2.5, 20/4.0, 21/1.5, 27/3....","[M-3, m-2, m2, m-2, M-3, P1, M-2]","[140.0, 144.0, 147.0, 158.0, 161.0, 212.0, 216...","[(-3, -2, -2, -2), (-2, -2, -2, -2)]","[4.0, 3.0, 11.0, 3.0, 51.0, 4.0, 3.0]","[[Superius], [Superius], [Superius], [Superius...",FUGA,8,True,0.0,None,7
12,Josquin Des Prés,Ave Maria,195.0,"[25/2.5, 31/3.0, 32/1.0, 35/3.0, 36/1.0]","[m9, m-6, M-3, m-6]","[195.0, 244.0, 248.0, 276.0, 280.0]","[(2, 2, 2, -2)]","[49.0, 4.0, 28.0, 4.0]","[Altus, [Superius], Altus, Tenor, Bassus]",FUGA,5,False,0.0,None,2
16,Josquin Des Prés,Ave Maria,316.0,"[40/3.0, 44/3.0, 44/4.0, 46/2.0, 46/3.0, 46/4....","[P5, m-10, m6, M6, M-10, M2]","[316.0, 348.0, 350.0, 362.0, 364.0, 366.0, 402.0]","[(3, 2, 2, -3), (2, 2, 2, -3)]","[32.0, 2.0, 12.0, 2.0, 2.0, 36.0]","[Altus, [Superius], Bassus, Tenor, [Superius],...",FUGA,7,True,1.0,Tenor,3
17,Josquin Des Prés,Ave Maria,332.0,"[42/3.0, 50/1.0, 50/2.0, 51/1.0, 51/2.0, 51/3.0]","[P5, M6, M-7, M6, M-6]","[332.0, 392.0, 394.0, 400.0, 402.0, 404.0]","[(-4, 2, -3, 2), (-2, 2, -3, 2), (-3, 2, -3, 2)]","[60.0, 2.0, 6.0, 2.0, 2.0]","[Bassus, Bassus, Tenor, Bassus, Tenor, Bassus]",FUGA,6,True,0.0,None,2
19,Josquin Des Prés,Ave Maria,344.0,"[44/1.0, 45/4.0, 46/1.0, 47/4.0, 48/1.0, 50/1.0]","[M-10, M6, m6, P-5, M6, M-9]","[344.0, 344.0, 358.0, 360.0, 374.0, 376.0, 392.0]","[(-2, 2, 2, 2), (-3, 2, 2, 2)]","[0.0, 14.0, 2.0, 14.0, 2.0, 16.0]","[[Superius], Bassus, Tenor, [Superius], Tenor,...",FUGA,7,True,1.0,Bassus,3
27,Josquin Des Prés,Ave Maria,392.0,"[50/1.0, 50/2.0, 51/1.0, 51/2.0, 51/3.0]","[M6, M-7, M6, M-6]","[392.0, 394.0, 400.0, 402.0, 404.0]","[(-2, 2, -3, 2), (-3, 2, -3, 2)]","[2.0, 6.0, 2.0, 2.0]","[Bassus, Tenor, Bassus, Tenor, Bassus]",FUGA,5,True,0.0,None,1
30,Josquin Des Prés,Ave Maria,398.0,"[50/4.0, 51/4.0, 53/2.0, 56/3.0, 57/3.0, 61/3....","[m-2, M-6, P5, m-7, M-2, m-7, P8, P8, M-7, P1,...","[398.0, 406.0, 418.0, 444.0, 452.0, 484.0, 492...","[(-2, -2, -2, 2), (-3, -2, -2, 2)]","[8.0, 12.0, 26.0, 8.0, 32.0, 8.0, 55.0, 51.0, ...","[[Superius], [Superius], Altus, [Superius], Al...",FUGA,14,True,0.0,None,11
33,Josquin Des Prés,Ave Maria,405.0,"[51/3.5, 55/4.0, 56/2.0, 60/4.0, 61/2.0]","[M9, P-5, P-4, P-5]","[405.0, 438.0, 442.0, 478.0, 482.0]","[(2, 2, -3, -2)]","[33.0, 4.0, 36.0, 4.0]","[Altus, [Superius], Altus, Tenor, Bassus]",FUGA,5,False,0.0,None,4
37,Josquin Des Prés,Ave Maria,448.0,"[57/1.0, 62/1.0, 69/1.0, 74/2.0, 75/3.0, 84/1....","[P-8, m7, P1, M2, P1, P4, m-2, M-10, P1, m7, P...","[448.0, 488.0, 544.0, 586.0, 596.0, 664.0, 672...","[(-3, -2, -2, -2), (-2, -2, -2, -2)]","[40.0, 56.0, 42.0, 10.0, 68.0, 8.0, 4.0, 28.0,...","[Altus, Bassus, Altus, Tenor, Tenor, Altus, [S...",FUGA,28,True,0.0,None,20


### Different Data Types Require Different Methods

Some fields (like composer or title) are simply **strings** of characters.  But the data in **Melodic_Entry_Intervals** (for instance) are **lists**, and as such we need to convert these to **strings** before we can search within them.

This is done by **applying a function** to all items in the column.  Here we **map** each item in the list to a 'string', then join those strings together as a single string, and update the column accordingly:

```
p_types["Melodic_Entry_Intervals"] = p_types["Melodic_Entry_Intervals"].apply(lambda x: ', '.join(map(str, x))).copy()
```

Now we can use `str.contains()` to find subpatterns within the Melodic Entry Intervals:

```
patterns_with_5 = p_types[p_types["Melodic_Entry_Intervals"].str.contains("5")].copy()
```

And now it is also possible to **count the values of the sets of entries** as strings:

```
p_types["Melodic_Entry_Intervals"].value_counts().to_frame()
```

In [8]:
p_types["Melodic_Entry_Intervals"] = p_types["Melodic_Entry_Intervals"].apply(lambda x: ', '.join(map(str, x))).copy()


In [9]:
p_types

,Composer,Title,First_Offset,Measures_Beats,Melodic_Entry_Intervals,Offsets,Soggetti,Time_Entry_Intervals,Voices,Presentation_Type,Number_Entries,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps
0,Josquin Des Prés,Ave Maria,0.0,"[1/1.0, 3/1.0, 5/1.0, 7/1.0]","P-8, P1, P-8","[0.0, 16.0, 32.0, 48.0]","[(4, 2, 2, -3), (3, 2, 2, -3), (2, 2, 2, -3)]","[16.0, 16.0, 16.0]","[[Superius], Altus, Tenor, Bassus]",PEN,4,True,0.0,None,0
1,Josquin Des Prés,Ave Maria,56.0,"[8/1.0, 10/1.0, 12/1.0, 14/1.0]","P-8, P1, P-8","[56.0, 72.0, 88.0, 104.0]","[(-2, -2, -2, 2), (-3, -2, -2, 2)]","[16.0, 16.0, 16.0]","[[Superius], Altus, Tenor, Bassus]",PEN,4,True,0.0,None,3
2,Josquin Des Prés,Ave Maria,62.0,"[8/4.0, 10/4.0, 12/4.0]","P-8, P1","[62.0, 78.0, 94.0]","[(-2, -2, 2, -2)]","[16.0, 16.0]","[[Superius], Altus, Tenor]",PEN,3,False,0.0,None,2
3,Josquin Des Prés,Ave Maria,64.0,"[9/1.0, 11/1.0, 13/1.0]","P-8, P1","[64.0, 80.0, 96.0]","[(-2, 2, -2, 4)]","[16.0, 16.0]","[[Superius], Altus, Tenor]",PEN,3,False,0.0,None,2
4,Josquin Des Prés,Ave Maria,68.0,"[9/3.0, 11/3.0, 13/3.0]","P-8, P1","[68.0, 84.0, 100.0]","[(2, -2, 4, -2), (3, -2, 4, -2)]","[16.0, 16.0]","[[Superius], Altus, Tenor]",PEN,3,True,0.0,None,2
5,Josquin Des Prés,Ave Maria,72.0,"[10/1.0, 12/1.0, 14/1.0]","P-8, P1","[72.0, 88.0, 104.0]","[(-2, 4, -2, -2), (-3, 4, -2, -2)]","[16.0, 16.0]","[[Superius], Altus, Tenor]",PEN,3,True,0.0,None,2
6,Josquin Des Prés,Ave Maria,74.0,"[10/2.0, 12/2.5, 14/2.0]","P-8, P1","[74.0, 91.0, 106.0]","[(4, -2, -2, 2), (2, -2, -2, 2), (3, -2, -2, 2)]","[17.0, 15.0]","[[Superius], Altus, Tenor]",FUGA,3,True,0.0,None,2
7,Josquin Des Prés,Ave Maria,76.0,"[10/3.0, 12/3.0, 14/3.0, 14/4.0, 23/1.0, 28/1....","P-8, P1, m-9, m9, P-8, M9","[76.0, 92.0, 108.0, 110.0, 176.0, 216.0, 221.0]","[(-2, -2, 2, 2), (-3, -2, 2, 2)]","[16.0, 16.0, 2.0, 66.0, 40.0, 5.0]","[[Superius], Altus, Tenor, Bassus, Altus, Bass...",FUGA,7,True,0.0,None,2
8,Josquin Des Prés,Ave Maria,140.0,"[18/3.0, 19/1.0, 19/2.5, 20/4.0, 21/1.5, 27/3....","M-3, m-2, m2, m-2, M-3, P1, M-2","[140.0, 144.0, 147.0, 158.0, 161.0, 212.0, 216...","[(-3, -2, -2, -2), (-2, -2, -2, -2)]","[4.0, 3.0, 11.0, 3.0, 51.0, 4.0, 3.0]","[[Superius], [Superius], [Superius], [Superius...",FUGA,8,True,0.0,None,7
9,Josquin Des Prés,Ave Maria,156.0,"[20/3.0, 27/2.5]",M-3,"[156.0, 211.0]","[(6, -2, -2, -2), (5, -2, -2, -2), (4, -2, -2,...",[55.0],"[[Superius], Altus]",FUGA,2,True,0.0,None,1


In [10]:
patterns_with_5 = p_types[p_types["Melodic_Entry_Intervals"].str.contains("m-7, m6")]
patterns_with_5

,Composer,Title,First_Offset,Measures_Beats,Melodic_Entry_Intervals,Offsets,Soggetti,Time_Entry_Intervals,Voices,Presentation_Type,Number_Entries,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps
26,Josquin Des Prés,Ave Maria,388.0,"[49/3.0, 49/4.0, 50/4.0, 51/1.0]","m6, m-7, m6","[388.0, 390.0, 398.0, 400.0]","[(2, -2, 2, -3)]","[2.0, 8.0, 2.0]","[Bassus, Tenor, Bassus, Tenor]",ID,4,False,0.0,None,2


In [11]:
p_types["Melodic_Entry_Intervals"].value_counts().to_frame()

,Melodic_Entry_Intervals
P-5,13
P-8,7
P1,6
"P-8, P1",5
"P-8, P8, P-8",4
"P-5, P-4, P-5",3
m6,3
M-3,3
"P-8, P1, P-8",2
"M6, P-5, M6",1


## Save to your folder of CSV's here in the Jupyter Hub
* You can save as CSV, or as Excel.
* You will then need to download this to your computer to view it properly
* Note that in the following part of the code below, you will need to give your file a name:


```
saved_csv/**file_name**.xlsx
```

In [13]:
writer = pd.ExcelWriter('saved_csv/file_name.xlsx', engine='xlsxwriter')
p_types.to_excel(writer, sheet_name='Sheet1')
writer.save()

In [14]:
p_types.to_csv('saved_csv/Verdelot_PTypes_MW_7_28_2022.csv')